## Validation of Feature Importance with Chi-Square test 

In [5]:
import pandas as pd
from scipy.stats import chi2_contingency


In [4]:
df = pd.read_csv("../Data/Cleaned/customer_churn_dataset.csv")
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
2,2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3,3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0


## Chi-Square test

In [7]:
import numpy as np

df["Issue_Level"] = np.where(
    df["Support Calls"] <= 2,
    "Low Issues",
    np.where(df["Support Calls"] <= 4,
             "Medium Issues",
             "High Issues")
)

df["Delay_Level"] = np.where(
    df["Payment Delay"] <= 15,
    "Low Delay",
    np.where(df["Payment Delay"] <= 20,
             "Medium Delay",
             "High Delay")
)

df["Spend_Level"] = np.where(
    df["Total Spend"] <= 508,
    "Low Spend",
    "High Spend"
)

In [14]:
from scipy.stats import chi2_contingency

features = [
    "Issue_Level",
    "Delay_Level",
    "Spend_Level",
    "Contract Length"
]

results = []

for feature in features:

    table = pd.crosstab(
        df[feature],
        df["Churn"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    results.append([
        feature,
        chi2,
        p
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Chi_Square",
        "P_Value"
    ]
)

results_df

,Feature,Chi_Square,P_Value
0,Issue_Level,152535.692764,0.0
1,Delay_Level,87480.684564,0.0
2,Spend_Level,94491.019220,0.0
3,Contract Length,67861.646650,0.0


#### Observation and Conclusion
1. The p value for all 4 most important feature is 0 which rejects the independance of these feature with customers churn.

## Cramér's V.

In [15]:
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(table):

    chi2 = chi2_contingency(table)[0]

    n = table.sum().sum()

    r, k = table.shape

    return np.sqrt(
        chi2 / (n * min(r - 1, k - 1))
    )

In [17]:
features = [
    "Issue_Level",
    "Delay_Level",
    "Spend_Level",
    "Contract Length"
]

results = []

for feature in features:

    table = pd.crosstab(
        df[feature],
        df["Churn"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    cv = cramers_v(table)

    results.append([
        feature,
        chi2,
        p,
        cv
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Chi_Square",
        "P_Value",
        "Cramers_V"
    ]
)

results_df.sort_values(
    by="Cramers_V",
    ascending=False
)

,Feature,Chi_Square,P_Value,Cramers_V
0,Issue_Level,152535.692764,0.0,0.549479
2,Spend_Level,94491.019220,0.0,0.432475
1,Delay_Level,87480.684564,0.0,0.416123
3,Contract Length,67861.646650,0.0,0.366503


#### Observation and Conclusion

- The Chi-Square test of independance can check the independance of features with customer churn but it cannot tell how strong is association with churn due to it's bias with sample size
- Cramér's V removes the effect of sample size as we got Cramér's V between 0.36 to 0.54 which suggests strong association with customer churn